In [1]:
import spotipy
import os
from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.oauth2 import SpotifyOAuth
from pprint import pprint
import pandas as pd
from dotenv import load_dotenv
import random

In [2]:
'''
dotenv file should contain:
SPOTIPY_CLIENT_ID = "your client id"
SPOTIPY_CLIENT_SECRET = "your client secret"  
SPOTIPY_REDIRECT_URI = "http://localhost:8888/callback"
'''
load_dotenv("C:/apis/.env") # path to your dotenv file
client_id = os.getenv("SPOTIPY_CLIENT_ID")
client_secret = os.getenv("SPOTIPY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIPY_REDIRECT_URI")

# This masks your secret keys before printing them, in case you are sharing this notebook:
def mask_secret(unmasked_chars, secret):
    masked_token = secret[:unmasked_chars] + '*' * (len(secret) - unmasked_chars*2) + secret[-unmasked_chars:]
    return masked_token

print(f"SPOTIPY_CLIENT_ID: {mask_secret(4, client_id)}")
print(f"SPOTIPY_CLIENT_SECRET: {mask_secret(4, client_secret)}")
print(f"SPOTIPY_REDIRECT_URI: {redirect_uri}")

SPOTIPY_CLIENT_ID: 3068************************2505
SPOTIPY_CLIENT_SECRET: 9945************************2daa
SPOTIPY_REDIRECT_URI: http://localhost:8888/callback


## Gathering all playlists

Checking all my playlists:

In [3]:
#sp = spotipy.Spotify(client_credentials_manager=SpotifyClientCredentials())
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=os.getenv("SPOTIPY_CLIENT_ID"),
                                               client_secret=os.getenv("SPOTIPY_CLIENT_SECRET"),
                                               redirect_uri=os.getenv("SPOTIPY_REDIRECT_URI"),
                                               scope="playlist-modify-public playlist-modify-private"))

user_id = "duhbeed"
if not user_id:
    raise ValueError("Spotify username is required.")

try:
    results = sp.user_playlists(user_id)
except spotipy.exceptions.SpotifyException as exc:
    print(f"Failed to fetch playlists: {exc}")
else:
    if not results["items"]:
        print("No public playlists found.")
    else:
        while True:
            for playlist in results["items"]:
                print(f"{playlist['name']} ({playlist['tracks']['total']} tracks)")
            if results["next"]:
                results = sp.next(results)
            else:
                break

⌛ Mad Cool 2025 - Jueves (orden horario) ⌛ (100 tracks)
My 2024 Playlist in a Bottle (8 tracks)
Best of 2024 (2/3) | Electronic & Hip-Hop (18 tracks)
Best of 2024 (1/3) | (Mostly) Rock (17 tracks)
Lo Mejor de 2024 (3/3) | de España y/o en castellano (15 tracks)
Dogs of TikTok, YouTube and Instagram (reddgr.com) - sorted by track popularity (43 tracks)
Low Festival 2024  (Sábado) 🌊 ¡Orden horario! ⏰ (100 tracks)
Low Festival 2024 🌊 (Domingo) ¡Orden horario! ⏰ (Domingo) (100 tracks)
Low Festival 2024 🌊 ¡Orden horario! ⏰ (Viernes) (100 tracks)
Talking to Chatbots (Reddgr) (International Playlist) (20 tracks)
Colección de podcasts de Reddgr (17 tracks)
Tomavistas 2024 (orden horario) (146 tracks)
Bands and Artists I've Seen Live (Sorted by artist popularity)  (478 tracks)
Reddgr Curated Podcasts (35 tracks)
Dogs of TikTok, YouTube and Instagram (reddgr.com) (44 tracks)
Talking to Chatbots (TTCB) (15 tracks)
DCODE 2022 by David (112 tracks)
Mad Cool 2022 (Jueves) (121 tracks)
2022 (55 track

Full dataframe with playlist metadata:

In [7]:
from html import unescape

playlist_items = []
page = sp.user_playlists(user_id, limit=50)
while page:
    playlist_items.extend(page.get("items", []))
    page = sp.next(page) if page.get("next") else None

column_map = {
    "name": "name",
    "tracks.total": "tracks_total",
    "description": "description",
    "id": "playlist_id",
    "snapshot_id": "snapshot_id",
    "external_urls.spotify": "external_url",
    "uri": "uri",
    "primary_color": "primary_color",
    "owner.display_name": "owner_display_name",
    "owner.id": "owner_id",
    "public": "public",
    "collaborative": "collaborative",
}

if playlist_items:
    playlists_raw = pd.json_normalize(playlist_items)
    available_cols = [col for col in column_map if col in playlists_raw.columns]
    playlists_df = (
        playlists_raw[available_cols]
        .rename(columns={col: column_map[col] for col in available_cols})
    )
    playlists_df = playlists_df[[column_map[col] for col in available_cols]]
else:
    playlists_df = pd.DataFrame(columns=list(column_map.values()))


playlists_df["description"] = playlists_df["description"].map(
    lambda value: unescape(value) if isinstance(value, str) else value
)

playlists_df.to_pickle("pkl/reddgr_playlists.pkl")
playlists_df.to_csv("csv/reddgr_playlists.csv", index=False)

display(playlists_df)

,name,tracks_total,description,playlist_id,snapshot_id,external_url,uri,primary_color,owner_display_name,owner_id,public,collaborative
0,⌛ Mad Cool 2025 - Jueves (orden horario) ⌛,100,100 pistas para preparar el primer día de Mad ...,0acg8XNM0LxaTeSan2U50T,AAAArqHP41rl5rCp+ib7PEnauUlV+gpj,https://open.spotify.com/playlist/0acg8XNM0Lxa...,spotify:playlist:0acg8XNM0LxaTeSan2U50T,None,David G. R. | reddgr,duhbeed,True,False
1,My 2024 Playlist in a Bottle,8,A musical time capsule from the past has been ...,1x5Tv4ITB7rq0OPcUB0NEo,AAAAA7aIkhTuCJ9n9Lq8DK1+XwUBCz3o,https://open.spotify.com/playlist/1x5Tv4ITB7rq...,spotify:playlist:1x5Tv4ITB7rq0OPcUB0NEo,None,David G. R. | reddgr,duhbeed,True,False
2,Best of 2024 (2/3) | Electronic & Hip-Hop,18,"Every year's curated playlists, to be listened...",2Bqun7K2S7cAksiUFCQTEm,AAAARHBDRPQyajXV3u4Km8mNKaKRCOoS,https://open.spotify.com/playlist/2Bqun7K2S7cA...,spotify:playlist:2Bqun7K2S7cAksiUFCQTEm,None,David G. R. | reddgr,duhbeed,True,False
3,Best of 2024 (1/3) | (Mostly) Rock,17,"Every year's curated playlists, to be listened...",4AA6jg6T0PbzkVJZKyI5X9,AAAAU5ugERkxM+m+7k0rQJpTmzXlWlaz,https://open.spotify.com/playlist/4AA6jg6T0Pbz...,spotify:playlist:4AA6jg6T0PbzkVJZKyI5X9,None,David G. R. | reddgr,duhbeed,True,False
4,Lo Mejor de 2024 (3/3) | de España y/o en cast...,15,Mi sesión de canciones en castellano y de arti...,1LyCSbuOeqglulGBKP8vSZ,AAAAVf8WUm9jTLDdOsdAdAumSdFK86qP,https://open.spotify.com/playlist/1LyCSbuOeqgl...,spotify:playlist:1LyCSbuOeqglulGBKP8vSZ,None,David G. R. | reddgr,duhbeed,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
148,Totally Random Playlist,20,,0AMHQuovdwMQ36DQip0UpQ,AAAAV/jDxNUkR+/zNbkrCdNguzd06UDz,https://open.spotify.com/playlist/0AMHQuovdwMQ...,spotify:playlist:0AMHQuovdwMQ36DQip0UpQ,None,David G. R. | reddgr,duhbeed,True,False
149,Video Games,10,,0RGDLwNwo4p80AZgms2ZsE,AAAAD9ZjUNOEJc/sSg+Naa7Cv6/oAZoJ,https://open.spotify.com/playlist/0RGDLwNwo4p8...,spotify:playlist:0RGDLwNwo4p80AZgms2ZsE,None,David G. R. | reddgr,duhbeed,True,False
150,Violence,11,,6AUxha1PnIg6mvvklt0j6n,AAAAD75grCTf+eBiHcDpAZ/bJTKw9g26,https://open.spotify.com/playlist/6AUxha1PnIg6...,spotify:playlist:6AUxha1PnIg6mvvklt0j6n,None,David G. R. | reddgr,duhbeed,True,False
151,Weezer 10,10,,0erlbFfIKEV4eMJEtKhuoV,AAAAHSKE24JOgaiS2tjbSwNI3FtEm/sr,https://open.spotify.com/playlist/0erlbFfIKEV4...,spotify:playlist:0erlbFfIKEV4eMJEtKhuoV,None,David G. R. | reddgr,duhbeed,True,False


In [10]:
playlists_df.sample(5)

,name,tracks_total,description,playlist_id,snapshot_id,external_url,uri,primary_color,owner_display_name,owner_id,public,collaborative
54,Best of 2017 (2/6) (Indie/Alternative),18,,7ehiBKzNQoxj7wtQNXmhad,AAAAi/XuiHzcFYHsyB3yEe56Ax7oypvy,https://open.spotify.com/playlist/7ehiBKzNQoxj...,spotify:playlist:7ehiBKzNQoxj7wtQNXmhad,None,David G. R. | reddgr,duhbeed,True,False
39,Best of 2019 (3/6),18,"My 100-song selection of every year, split int...",0TPuVun8RAJiw6MXRB4590,AAAAkXgbVM0LLPd7Yp+VloT6QVrZ/Q1G,https://open.spotify.com/playlist/0TPuVun8RAJi...,spotify:playlist:0TPuVun8RAJiw6MXRB4590,None,David G. R. | reddgr,duhbeed,True,False
87,SOS 4.8 2014,39,,1gJH0TrY1WQw7OQOMR88ZJ,AAAAYDMc4JX0CzgpeGF6FAK59rMKu7De,https://open.spotify.com/playlist/1gJH0TrY1WQw...,spotify:playlist:1gJH0TrY1WQw7OQOMR88ZJ,None,David G. R. | reddgr,duhbeed,True,False
75,Best of 2015 (2/6) (Alternative/Other),17,,6CsLdv6uzrhOZA6dFSWSat,AAAAbkdB+49xKTrBGcuP6wpL+7cm8TVX,https://open.spotify.com/playlist/6CsLdv6uzrhO...,spotify:playlist:6CsLdv6uzrhOZA6dFSWSat,None,David G. R. | reddgr,duhbeed,True,False
109,1985,23,,5x8JXqhNJhJdbDwUXXUgFE,AAAAPXR7ftY16J5GITf5DRoXShHAjj9U,https://open.spotify.com/playlist/5x8JXqhNJhJd...,spotify:playlist:5x8JXqhNJhJdbDwUXXUgFE,None,David G. R. | reddgr,duhbeed,True,False


In [11]:
bestof_year_playlists_df = (
    playlists_df[playlists_df["name"].str.lower().str.startswith(("best of", "lo mejor de"))]
    .reset_index(drop=True)
)

bestof_year_playlists_df = bestof_year_playlists_df.assign(
    year=pd.to_numeric(
        bestof_year_playlists_df["name"].str.extract(r"\b(2[0-9]{3})\b", expand=False),
        errors="coerce"
    )
)
main_cols = ["name", "year", "tracks_total", "description"]
cols = main_cols + [col for col in bestof_year_playlists_df.columns if col not in main_cols]
bestof_year_playlists_df = bestof_year_playlists_df[cols]

bestof_year_playlists_df = bestof_year_playlists_df.sort_values("year", ascending=True, na_position="last")
display(bestof_year_playlists_df)

,name,year,tracks_total,description,playlist_id,snapshot_id,external_url,uri,primary_color,owner_display_name,owner_id,public,collaborative
19,Best of 2010,2010,102,,5iBmxKUnNKzEZoYXxHYC2r,AAABUvdiqXREh7R4u2TxzLIgINVccq2z,https://open.spotify.com/playlist/5iBmxKUnNKzE...,spotify:playlist:5iBmxKUnNKzEZoYXxHYC2r,None,David G. R. | reddgr,duhbeed,True,False
20,Best of 2011,2011,101,,2lMwz55DQSDpnNed34XFRa,AAACCO8tlBBDSkBWi+zh+FUIVowCy5pz,https://open.spotify.com/playlist/2lMwz55DQSDp...,spotify:playlist:2lMwz55DQSDpnNed34XFRa,None,David G. R. | reddgr,duhbeed,True,False
71,Best of 2012 (bonus tracks),2012,23,,4jVzmc44I1IfrvgB2gaCIV,AAABztHil5w7vv0K1kpcb/CTmBbaf6u/,https://open.spotify.com/playlist/4jVzmc44I1If...,spotify:playlist:4jVzmc44I1IfrvgB2gaCIV,None,David G. R. | reddgr,duhbeed,True,False
69,Best of 2012 (4/5) (electronic/hip-hop),2012,20,,73kk51cnXJvlsLp2LOi6wD,AAAAQvlfSNVRg/p7A6FQ20ae2iLiYYWA,https://open.spotify.com/playlist/73kk51cnXJvl...,spotify:playlist:73kk51cnXJvlsLp2LOi6wD,None,David G. R. | reddgr,duhbeed,True,False
68,Best of 2012 (3/5) (experimental/other),2012,20,,4KYIK7rzuEO96twgvOynvM,AAAAVUe3z/AAcc+u3FJm1qbZaOKUUM1F,https://open.spotify.com/playlist/4KYIK7rzuEO9...,spotify:playlist:4KYIK7rzuEO96twgvOynvM,None,David G. R. | reddgr,duhbeed,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,Best of 2023 (2/3),2023,18,"In this year's collection, I've selected 50 so...",4XOMosM5oObvrcQVFCamFj,AAAAN2HQJgGJ1y7a9q0OzXqWJ9a+kB5H,https://open.spotify.com/playlist/4XOMosM5oObv...,spotify:playlist:4XOMosM5oObvrcQVFCamFj,None,David G. R. | reddgr,duhbeed,True,False
3,Best of 2023 (1/3),2023,18,"In this year's collection, I've selected 50 so...",34wg9M1ElBgos2l6QGExCd,AAAAPCLp2VtHkE0rm+UTkhq9PEIEf1zL,https://open.spotify.com/playlist/34wg9M1ElBgo...,spotify:playlist:34wg9M1ElBgos2l6QGExCd,None,David G. R. | reddgr,duhbeed,True,False
2,Lo Mejor de 2024 (3/3) | de España y/o en cast...,2024,15,Mi sesión de canciones en castellano y de arti...,1LyCSbuOeqglulGBKP8vSZ,AAAAVf8WUm9jTLDdOsdAdAumSdFK86qP,https://open.spotify.com/playlist/1LyCSbuOeqgl...,spotify:playlist:1LyCSbuOeqglulGBKP8vSZ,None,David G. R. | reddgr,duhbeed,True,False
1,Best of 2024 (1/3) | (Mostly) Rock,2024,17,"Every year's curated playlists, to be listened...",4AA6jg6T0PbzkVJZKyI5X9,AAAAU5ugERkxM+m+7k0rQJpTmzXlWlaz,https://open.spotify.com/playlist/4AA6jg6T0Pbz...,spotify:playlist:4AA6jg6T0PbzkVJZKyI5X9,None,David G. R. | reddgr,duhbeed,True,False


## Working with playlists table

In [21]:
bestof_year_playlists_df = pd.read_pickle("pkl/reddgr_playlists.pkl")
display(bestof_year_playlists_df.sample(5))

,name,tracks_total,description,playlist_id,snapshot_id,external_url,uri,primary_color,owner_display_name,owner_id,public,collaborative
135,Primavera Sound 2012 - Thursday,39,,4BoepfPVGuFcVaGKGqjdEw,AAAAONgbKeJtOGzwYBVU/JFoWm/791yr,https://open.spotify.com/playlist/4BoepfPVGuFc...,spotify:playlist:4BoepfPVGuFcVaGKGqjdEw,None,David G. R. | reddgr,duhbeed,True,False
148,Totally Random Playlist,20,,0AMHQuovdwMQ36DQip0UpQ,AAAAV/jDxNUkR+/zNbkrCdNguzd06UDz,https://open.spotify.com/playlist/0AMHQuovdwMQ...,spotify:playlist:0AMHQuovdwMQ36DQip0UpQ,None,David G. R. | reddgr,duhbeed,True,False
129,Pitchfork: Top 200 Tracks of the 90s,163,,0TFokeJiiARIiXLdlib9g1,AAAAuadf5y9ezhi5y3vUlznFWcx/JWYF,https://open.spotify.com/playlist/0TFokeJiiARI...,spotify:playlist:0TFokeJiiARIiXLdlib9g1,None,Jonathan Hall,the_junes,True,False
144,States,15,,64YhtLiIOyGjyREhHE3gDO,AAAACSJT7IpKy9imwwlgEuAWYxM6Ktqj,https://open.spotify.com/playlist/64YhtLiIOyGj...,spotify:playlist:64YhtLiIOyGjyREhHE3gDO,None,David G. R. | reddgr,duhbeed,True,False
50,Mad Cool 2018,310,,1UpalvfEZYGGUDUT2ox4iy,AAABOEgAkqElh3gGnLI88mDh9pU+PrJu,https://open.spotify.com/playlist/1UpalvfEZYGG...,spotify:playlist:1UpalvfEZYGGUDUT2ox4iy,None,David G. R. | reddgr,duhbeed,True,False


DF for playlist:

In [22]:
target_playlist = bestof_year_playlists_df.iloc[random.randint(0, len(bestof_year_playlists_df) - 1)]

def format_duration(ms):
    if ms is None:
        return None
    minutes, seconds = divmod(int(ms) // 1000, 60)
    return f"{minutes}:{seconds:02d}"

tracks_data = []
offset = 0
position = 1
while True:
    response = sp.playlist_items(target_playlist["playlist_id"], offset=offset, limit=100)
    items = response.get("items", [])
    if not items:
        break
    for item in items:
        track = item.get("track")
        if not track or track.get("type") != "track":
            continue
        album = track.get("album") or {}
        tracks_data.append({
            "position": position,
            "artists": ", ".join(artist.get("name") for artist in track.get("artists", [])),
            "track_name": track.get("name"),
            "album_name": album.get("name"),
            "release_date": album.get("release_date"),
            "track_popularity": track.get("popularity"),
            "duration_ms": track.get("duration_ms"),
            "added_at": item.get("added_at"),         
            "duration_mmss": format_duration(track.get("duration_ms")),           
            "external_url": (track.get("external_urls") or {}).get("spotify"),
            "track_id": track.get("id"),
            "track_uri": track.get("uri"),
        })
        position += 1
    offset += len(items)
    if not response.get("next"):
        break

if tracks_data:
    playlist_tracks_df = pd.DataFrame(tracks_data)
    playlist_tracks_df["release_date"] = pd.to_datetime(playlist_tracks_df["release_date"], errors="coerce")
    playlist_tracks_df["added_at"] = pd.to_datetime(playlist_tracks_df["added_at"], errors="coerce")
else:
    playlist_tracks_df = pd.DataFrame(columns=[
        "position", "track_id", "track_uri", "track_name", "artists", "album_name",
        "release_date", "added_at", "duration_ms", "duration_mmss", "track_popularity", "external_url"
    ])

display(target_playlist[["name", "playlist_id", "tracks_total"]].to_frame().T)
display(playlist_tracks_df)

,name,playlist_id,tracks_total
82,Mad Cool 2016,3PhC3YAHgZumqLrx7xVLNO,200


,position,artists,track_name,album_name,release_date,track_popularity,duration_ms,added_at,duration_mmss,external_url,track_id,track_uri
0,1,Buffalo Springfield,Mr. Soul,Buffalo Springfield Again,1967-10-30,58,172400,2016-02-03 13:41:30+00:00,2:52,https://open.spotify.com/track/042v1NacbKJzCyi...,042v1NacbKJzCyi6nBme7T,spotify:track:042v1NacbKJzCyi6nBme7T
1,2,"Neil Young, Crazy Horse",Mideast Vacation,Lucky Thirteen,1993-01-11,0,263266,2016-02-03 13:45:31+00:00,4:23,https://open.spotify.com/track/3EOCwxf694FqQSZ...,3EOCwxf694FqQSZj4lsUgz,spotify:track:3EOCwxf694FqQSZj4lsUgz
2,3,The Prodigy,Breathe,The Fat of the Land,1997-06-30,1,334800,2016-02-03 13:46:15+00:00,5:34,https://open.spotify.com/track/5FCyoZGaN6nRod1...,5FCyoZGaN6nRod1oNDJIWZ,spotify:track:5FCyoZGaN6nRod1oNDJIWZ
3,4,The Prodigy,Voodoo People (Remastered),More Music for the Jilted Generation,NaT,0,387066,2016-02-03 13:46:39+00:00,6:27,https://open.spotify.com/track/1dWteEXxzhg5xGN...,1dWteEXxzhg5xGNZZ6dJQX,spotify:track:1dWteEXxzhg5xGNZZ6dJQX
4,5,The Prodigy,No Good (Start the Dance),Music for the Jilted Generation,1994-07-04,0,377893,2016-02-03 13:47:00+00:00,6:17,https://open.spotify.com/track/4kcBLVepesflQP3...,4kcBLVepesflQP345wKVeq,spotify:track:4kcBLVepesflQP345wKVeq
...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,León Benavente,Estado Provisional,León Benavente,2013-05-28,0,217706,2016-06-14 08:20:06+00:00,3:37,https://open.spotify.com/track/176Wqfo5PFq8tMt...,176Wqfo5PFq8tMtAzh5gIo,spotify:track:176Wqfo5PFq8tMtAzh5gIo
196,197,León Benavente,Las Ruinas,León Benavente,2013-05-28,0,225000,2016-06-14 08:20:06+00:00,3:45,https://open.spotify.com/track/26CzzDkBlxpJnDy...,26CzzDkBlxpJnDyQXYcB95,spotify:track:26CzzDkBlxpJnDyQXYcB95
197,198,León Benavente,Década,León Benavente,2013-05-28,0,292760,2016-06-14 08:20:06+00:00,4:52,https://open.spotify.com/track/2lshqSuyaBsrSLK...,2lshqSuyaBsrSLKh8Jg4Xa,spotify:track:2lshqSuyaBsrSLKh8Jg4Xa
198,199,León Benavente,Ser Brigada,León Benavente,2013-05-28,0,302813,2016-06-14 08:20:06+00:00,5:02,https://open.spotify.com/track/0SRMrCKFp0X8vrc...,0SRMrCKFp0X8vrcVfa2rP6,spotify:track:0SRMrCKFp0X8vrcVfa2rP6
